In [1]:
import pandas as pd
import numpy as np
import html
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv('./anime.csv')

In [3]:
print(df.shape)
print(df.head())
print(df.describe())
print(df.info)

(12294, 7)
   anime_id                              name  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                          Gintama°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre   type episodes  rating  \
0               Drama, Romance, School, Supernatural  Movie        1    9.37   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV       64    9.26   
2  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.25   
3                                   Sci-Fi, Thriller     TV       24    9.17   
4  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.16   

   members  
0   200630  
1   793665  
2   114262  
3   673572  
4   151266  
           anime_id        rating       members
count  12294.000000  12064.000000  1.229400e+04
mean   14058.221653      6.473902  1.80

In [4]:
print(df.dtypes)

anime_id      int64
name         object
genre        object
type         object
episodes     object
rating      float64
members       int64
dtype: object


dropping nulls

In [5]:
print(df.isnull().sum())

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64


In [6]:
df = df.dropna()

In [7]:
print(df.isnull().sum())
print(df.shape)

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64
(12017, 7)


In [8]:
print(df.duplicated().sum())

0


In [9]:
print(df['genre'].value_counts())

genre
Hentai                                 816
Comedy                                 521
Music                                  297
Kids                                   197
Comedy, Slice of Life                  174
                                      ... 
Action, Hentai, Mecha, Sci-Fi, Yaoi      1
Adventure, Fantasy, Hentai               1
Hentai, Horror, Yaoi                     1
Hentai, Space                            1
Drama, Hentai, Mystery, Romance          1
Name: count, Length: 3229, dtype: int64


df['name'] and df['genre'] into lowercase

In [10]:
df['name'] = df['name'].str.lower()
df['genre'] = df['genre'].str.lower()

turning eps into numeric

In [11]:
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')

In [12]:
df['rating'].isnull().sum()

np.int64(0)

used html.unescape to correct the names

In [13]:
df['name'] = df['name'].apply(html.unescape)

In [14]:
df['name'].head

<bound method NDFrame.head of 0                                           kimi no na wa.
1                         fullmetal alchemist: brotherhood
2                                                 gintama°
3                                              steins;gate
4                                                 gintama'
                               ...                        
12289         toushindai my lover: minami tai mecha-minami
12290                                          under world
12291                       violence gekiga david no hoshi
12292    violence gekiga shin david no hoshi: inma dens...
12293                     yasuji no pornorama: yacchimae!!
Name: name, Length: 12017, dtype: object>

maping all the other types to series and movie stays movie

In [15]:
df['type'].value_counts()

type
TV         3668
OVA        3284
Movie      2259
Special    1670
ONA         648
Music       488
Name: count, dtype: int64

In [16]:
df['type'] = df['type'].map({'TV':'series', 'OVA':'series', 'Special':'series', 'ONA':'series', 'Music':'series', 'Movie':'movie'})

In [17]:
df['type'].value_counts()

type
series    9758
movie     2259
Name: count, dtype: int64

creating eps_tag

In [18]:
def eps_res(rows):
    if (rows['episodes']==1): return 'movie'
    elif (rows['episodes']<=13): return 'short_series'
    elif (rows['episodes']<=26): return 'medium_series'
    else: return 'long_series'

df['eps_tag'] = df.apply(eps_res, axis=1)



In [19]:
df.columns

Index(['anime_id', 'name', 'genre', 'type', 'episodes', 'rating', 'members',
       'eps_tag'],
      dtype='object')

In [20]:
df['eps_tag'].value_counts()

eps_tag
movie            5571
short_series     4027
long_series      1331
medium_series    1088
Name: count, dtype: int64

creating tfidf_input 

In [21]:
df['tfidf_input'] = df['genre'].str.lower()

In [22]:
df['tfidf_input'] = df['tfidf_input'].str.replace(',',' ')
df['tfidf_input'] = df['tfidf_input'].str.cat(df['eps_tag'], sep=" ")


In [23]:
df['type'].value_counts()

type
series    9758
movie     2259
Name: count, dtype: int64

In [24]:
df['tfidf_input'].head

<bound method NDFrame.head of 0               drama  romance  school  supernatural movie
1        action  adventure  drama  fantasy  magic  mili...
2        action  comedy  historical  parody  samurai  s...
3                           sci-fi  thriller medium_series
4        action  comedy  historical  parody  samurai  s...
                               ...                        
12289                                         hentai movie
12290                                         hentai movie
12291                                  hentai short_series
12292                                         hentai movie
12293                                         hentai movie
Name: tfidf_input, Length: 12017, dtype: object>

model implementation

In [25]:
tf = TfidfVectorizer()
tfidf_matrix = tf.fit_transform(df['tfidf_input'])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [27]:
print(tfidf_matrix.shape)
print(cosine_sim.shape)

(12017, 51)
(12017, 12017)


# Recommendation model

In [40]:
name_to_idx = pd.Series(df.index, index=df["name"])


def recommendation(anime):
    
    try:
        if anime in name_to_idx:
            
            idx = name_to_idx[anime]
            scores = list(enumerate(cosine_sim[idx]))
        else:
            query_vec = tf.transform([anime])
            scores = list(enumerate(cosine_similarity(query_vec, tfidf_matrix)[0]))
        
        scores = sorted(scores, key=lambda x: x[1], reverse=True)    
        top5 = scores[1:6]
        results = []
        for i, score in top5:
            results.append({
                "name"  : df.iloc[i]["name"],
                "genre" : df.iloc[i]["genre"],
                "rating": float(df.iloc[i]["rating"]),
                "type"  : df.iloc[i]["type"]
            })   

        return results

    except KeyError:
        
        return print('Please enter the correct input...') 

anime = str.lower((input("please enter the anime genre or the simialr anime you want to watch: ")))
result = recommendation(anime)
print(result)

please enter the anime genre or the simialr anime you want to watch: action
[{'name': "flag director's edition: issenman no kufura no kiroku", 'genre': 'action', 'rating': 7.01, 'type': 'movie'}, {'name': 'fuuma no kojirou: fuuma hanran-hen', 'genre': 'action', 'rating': 6.93, 'type': 'series'}, {'name': 'ghost in the shell arise episode: [.jp]', 'genre': 'action', 'rating': 6.68, 'type': 'series'}, {'name': 'sonic: night of the werehog', 'genre': 'action', 'rating': 6.63, 'type': 'series'}, {'name': 'break blade: virgins war', 'genre': 'action', 'rating': 6.58, 'type': 'series'}]
